# Phase 3A — Leakage-safe baseline model training in Google Colab

This notebook trains four small baseline classifiers on the labeled Kaggle credit-score data. It uses the repository's customer-grouped split, deterministic cleaning, extreme-value handling, and development-fitted preprocessing.

**Safety boundary:** only development data teaches preprocessing and models. Validation is used for comparison. The final-test partition stays sealed, Kaggle `test.csv` is never requested, and no model or transformed dataset is saved. These baselines are learning experiments—not production-ready systems.

## 1. Mount Google Drive

**What:** connect this Colab runtime to your Drive. **Why:** `train.csv` remains in your private Drive instead of the Git repository. **Expected output:** a successful mount message. **Check before continuing:** confirm Colab shows `Mounted at /content/drive`.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Configuration

**What:** define the dataset, repository, branch, checksum, class order, and random seed in one visible place. **Why:** these values should be easy to audit and change deliberately. **Expected output:** a short configuration summary without any customer data. **Check:** the dataset path must point to your Drive copy of labeled `train.csv`.

In [ ]:
from pathlib import Path

DATASET_PATH = Path('/content/drive/MyDrive/Credit-Scoring-Model/data/raw/kaggle_credit_score/train.csv')
EXPECTED_SHA256 = 'D2EBCC056A64C48710B1AEB96777155835D372D7AD202529F64666011D214DA0'
REPOSITORY_URL = 'https://github.com/MaryamCodeHub/Credit-Scoring-Model.git'
REPOSITORY_DIR = Path('/content/Credit-Scoring-Model')
BRANCH = 'model-improvement-v2'
RANDOM_STATE = 42
CLASS_ORDER = ['Poor', 'Standard', 'Good']

print(f'Dataset: {DATASET_PATH}')
print(f'Repository branch: {BRANCH}')
print(f'Class order: {CLASS_ORDER}')

## 3. Verify the labeled dataset before reading it

**What:** check that `train.csv` exists and calculate its SHA-256 hash. **Why:** a missing, incomplete, or different file would make results incomparable. **Expected output:** `Dataset file and SHA-256 verified.` **Check:** do not continue if this cell raises an error. It intentionally stops on a wrong path or hash.

In [ ]:
import hashlib

if not DATASET_PATH.is_file():
    raise FileNotFoundError(
        f'Labeled train.csv was not found at {DATASET_PATH}. '
        'Upload it to the configured Google Drive path before continuing.'
    )

digest = hashlib.sha256()
with DATASET_PATH.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        digest.update(chunk)
actual_sha256 = digest.hexdigest().upper()

if actual_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        'SHA-256 mismatch. Stop: this is not the audited train.csv. '
        f'Expected {EXPECTED_SHA256}, received {actual_sha256}.'
    )

print('Dataset file and SHA-256 verified.')

## 4. Clone or safely update the repository

**What:** clone the GitHub repository into the temporary Colab runtime, or fast-forward an existing clone, then explicitly check out `model-improvement-v2`. **Why:** training must use the reviewed repository code. **Expected output:** the active branch and commit hash. **Check:** the printed branch must be exactly `model-improvement-v2`.

In [ ]:
import subprocess

def run_command(arguments, cwd=None):
    return subprocess.run(
        arguments,
        cwd=cwd,
        check=True,
        text=True,
        capture_output=True,
    )

if (REPOSITORY_DIR / '.git').is_dir():
    run_command(['git', 'fetch', 'origin', BRANCH], cwd=REPOSITORY_DIR)
    run_command(['git', 'checkout', BRANCH], cwd=REPOSITORY_DIR)
    run_command(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPOSITORY_DIR)
else:
    run_command([
        'git', 'clone', '--branch', BRANCH, '--single-branch',
        REPOSITORY_URL, str(REPOSITORY_DIR),
    ])

active_branch = run_command(
    ['git', 'branch', '--show-current'], cwd=REPOSITORY_DIR
).stdout.strip()
commit_hash = run_command(
    ['git', 'rev-parse', 'HEAD'], cwd=REPOSITORY_DIR
).stdout.strip()
if active_branch != BRANCH:
    raise RuntimeError(f'Wrong branch: expected {BRANCH}, found {active_branch}.')

print(f'Active branch: {active_branch}')
print(f'Commit: {commit_hash}')

## 5. Install only the runtime packages needed here

**What:** install the repository's NumPy, pandas, and scikit-learn versions plus Matplotlib for confusion matrices. **Why:** matching reviewed versions improves reproducibility; API, SMOTE, tuning, GPU, and deep-learning packages are unnecessary. **Expected output:** pip completes without a dependency error. **Check:** restart the Colab runtime only if pip explicitly asks, then rerun from the top.

In [ ]:
import sys

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '--quiet',
    'numpy==2.4.4',
    'pandas==3.0.2',
    'scikit-learn==1.8.0',
    'matplotlib>=3.8,<4',
])
print('Required Colab dependencies are installed.')

## 6. Import the reviewed workflow and evaluation tools

**What:** import repository functions and the four baseline estimators. **Why:** this notebook should call the tested implementation instead of duplicating cleaning logic. **Expected output:** package versions and `Imports completed.` **Check:** resolve any import error before loading data; warnings are not suppressed.

In [ ]:
import os
import time

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.tree import DecisionTreeClassifier

os.chdir(REPOSITORY_DIR)
if str(REPOSITORY_DIR) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_DIR))

from src.data_cleaning import clean_credit_data
from src.data_splitting import split_by_customer
from src.model_preprocessing import (
    FORBIDDEN_COLUMNS,
    QUARANTINED_COLUMNS,
    build_model_preprocessor,
    separate_features_target_groups,
)

print(f'NumPy: {np.__version__}')
print(f'pandas: {pd.__version__}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'Matplotlib: {matplotlib.__version__}')
print('Imports completed.')

## 7. Load labeled `train.csv` without displaying personal data

**What:** read the audited labeled CSV and validate its target classes. **Why:** schema checks catch accidental files early. **Expected output:** only the overall shape, column count, and target class names—never names, SSNs, IDs, customer IDs, or sample rows. **Check:** the classes must be Poor, Standard, and Good.

In [ ]:
raw_train = pd.read_csv(DATASET_PATH, low_memory=False)
required_columns = {'Customer_ID', 'Credit_Score'}
missing_columns = sorted(required_columns.difference(raw_train.columns))
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

observed_classes = set(raw_train['Credit_Score'].dropna().unique())
if observed_classes != set(CLASS_ORDER):
    raise ValueError(
        f'Unexpected target classes: {sorted(observed_classes)}; '
        f'expected {CLASS_ORDER}.'
    )

print(f'Labeled dataset shape: {raw_train.shape}')
print(f'Column count: {raw_train.shape[1]}')
print(f'Target classes: {CLASS_ORDER}')

## 8. Split by customer and seal the final test

**What:** assign every customer to development, validation, or final test using the repository's grouped splitter. **Why:** rows from one customer must never appear in multiple partitions. **Expected output:** development and validation sizes plus a sealed-final-test message. **Check:** all overlap assertions must pass. Final-test values are not summarized.

In [ ]:
partitions = split_by_customer(raw_train, random_state=RANDOM_STATE)

development_customers = set(partitions.development_train['Customer_ID'])
validation_customers = set(partitions.validation['Customer_ID'])
final_test_customers = set(partitions.final_test['Customer_ID'])
assert development_customers.isdisjoint(validation_customers)
assert development_customers.isdisjoint(final_test_customers)
assert validation_customers.isdisjoint(final_test_customers)

print(f'Development rows: {len(partitions.development_train):,}')
print(f'Validation rows: {len(partitions.validation):,}')
print('Final-test partition was created and is now sealed.')

del development_customers, validation_customers, final_test_customers

## 9. Apply deterministic cleaning independently

**What:** clean each partition with fixed parsing and validity rules. **Why:** independent calls prevent one partition from supplying values to another. **Expected output:** a completion message only. **Check:** no learned statistic is fitted here, and no cleaned CSV is saved.

In [ ]:
clean_development = clean_credit_data(partitions.development_train)
clean_validation = clean_credit_data(partitions.validation)
clean_final_test = clean_credit_data(partitions.final_test)

print('All three partitions were cleaned independently in memory.')

## 10. Separate approved features, target, and customer groups

**What:** use the repository feature contract to create `X`, `y`, and grouping keys. **Why:** identifiers, PII, target, and quarantined variables must not enter the model matrix. **Expected output:** development/validation feature shapes and a safety confirmation. **Check:** both forbidden-column assertions must pass; no identifier values are displayed.

In [ ]:
development_inputs = separate_features_target_groups(clean_development)
validation_inputs = separate_features_target_groups(clean_validation)
_sealed_final_inputs = separate_features_target_groups(clean_final_test)

unsafe_columns = set(FORBIDDEN_COLUMNS) | set(QUARANTINED_COLUMNS)
assert unsafe_columns.isdisjoint(development_inputs.X.columns)
assert unsafe_columns.isdisjoint(validation_inputs.X.columns)
assert 'Credit_Score' not in development_inputs.X.columns

print(f'Development X shape: {development_inputs.X.shape}')
print(f'Validation X shape: {validation_inputs.X.shape}')
print('Forbidden, PII, target, and quarantined columns are absent from X.')
print('Final-test inputs remain sealed and will not be transformed or inspected.')

## 11. Fit preprocessing on development only

**What:** fit extreme-value handling, median imputation, and Occupation encoding using development data, then transform development and validation. **Why:** validation must not influence thresholds, medians, or category vocabulary. **Expected output:** matching transformed column counts and zero missing values. **Check:** no variable named `X_final_test_transformed` should exist.

In [ ]:
preprocessor = build_model_preprocessor()
preprocessor.fit(development_inputs.X)

X_development = preprocessor.transform(development_inputs.X)
X_validation = preprocessor.transform(validation_inputs.X)
y_development = development_inputs.y.copy()
y_validation = validation_inputs.y.copy()
feature_names = preprocessor.get_feature_names_out()

assert X_development.shape[1] == X_validation.shape[1] == len(feature_names)
assert not np.isnan(X_development.astype(float)).any()
assert not np.isnan(X_validation.astype(float)).any()
assert unsafe_columns.isdisjoint(feature_names)
assert 'X_final_test_transformed' not in globals()

print(f'Transformed development shape: {X_development.shape}')
print(f'Transformed validation shape: {X_validation.shape}')
print(f'Output feature count: {len(feature_names)}')
print('Preprocessor fitted on development only; validation received transform() only.')
print('Final test remains sealed: no transform, prediction, or evaluation was performed.')

## 12. Define four resource-conscious baseline models

**What:** configure a Dummy baseline, Logistic Regression, a conservative Decision Tree, and a lightweight Random Forest. **Why:** this gives a progression from a no-skill reference to linear and nonlinear models without tuning.

**Non-default choices:** Logistic Regression gets `max_iter=1000` to reduce convergence risk. The tree uses depth 8, at least 50 rows per leaf, and balanced class weights to limit memorization. Random Forest uses 100 trees (the allowed maximum), depth 12, at least 10 rows per leaf, balanced bootstrap weights, and at most two CPU workers. Every stochastic model uses seed 42. There is no SMOTE, scaling, tuning, cross-validation, GPU library, or deep learning.

**Expected output:** four model names. **Check:** Random Forest must show no more than 100 trees and `n_jobs=2`.

In [ ]:
models = {
    'DummyClassifier': DummyClassifier(strategy='most_frequent'),
    'LogisticRegression': LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
    ),
    'DecisionTreeClassifier': DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=50,
        class_weight='balanced',
        random_state=RANDOM_STATE,
    ),
    'RandomForestClassifier': RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=10,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
        n_jobs=2,
    ),
}

print('Models:')
for model_name in models:
    print(f'  - {model_name}')

## 13. Define validation-only evaluation

**What:** create a helper that times training and validation prediction, then calculates all requested metrics in the fixed class order. **Why:** one shared function makes comparisons consistent. **Expected output:** no output yet; this cell only defines the function. **Check:** predictions are made only for development and validation. `zero_division=0` is explicit because DummyClassifier never predicts two classes; warnings are otherwise not suppressed.

In [ ]:
def fit_and_evaluate(model_name, model):
    training_started = time.perf_counter()
    model.fit(X_development, y_development)
    training_seconds = time.perf_counter() - training_started

    development_predictions = model.predict(X_development)
    prediction_started = time.perf_counter()
    validation_predictions = model.predict(X_validation)
    prediction_seconds = time.perf_counter() - prediction_started

    train_macro_f1 = f1_score(
        y_development, development_predictions,
        labels=CLASS_ORDER, average='macro', zero_division=0,
    )
    validation_macro_f1 = f1_score(
        y_validation, validation_predictions,
        labels=CLASS_ORDER, average='macro', zero_division=0,
    )
    metrics = {
        'Model': model_name,
        'Validation Macro F1': validation_macro_f1,
        'Training Macro F1': train_macro_f1,
        'Train-Validation Gap': train_macro_f1 - validation_macro_f1,
        'Accuracy': accuracy_score(y_validation, validation_predictions),
        'Balanced Accuracy': balanced_accuracy_score(
            y_validation, validation_predictions
        ),
        'Weighted F1': f1_score(
            y_validation, validation_predictions,
            labels=CLASS_ORDER, average='weighted', zero_division=0,
        ),
        'Macro Precision': precision_score(
            y_validation, validation_predictions,
            labels=CLASS_ORDER, average='macro', zero_division=0,
        ),
        'Macro Recall': recall_score(
            y_validation, validation_predictions,
            labels=CLASS_ORDER, average='macro', zero_division=0,
        ),
        'Training Seconds': training_seconds,
        'Validation Prediction Seconds': prediction_seconds,
    }
    report = classification_report(
        y_validation,
        validation_predictions,
        labels=CLASS_ORDER,
        target_names=CLASS_ORDER,
        output_dict=True,
        zero_division=0,
    )
    matrix = confusion_matrix(
        y_validation, validation_predictions, labels=CLASS_ORDER
    )
    return metrics, report, matrix

## 14. Train baselines and evaluate validation only

**What:** fit each model on transformed development data and evaluate it on transformed validation data. **Why:** validation estimates performance on unseen customers while preserving the final test for a later phase. **Expected output:** completion and approximate timing for each model. **Check:** there is no final-test prediction and no model-saving call.

In [ ]:
comparison_rows = []
classification_reports = {}
confusion_matrices = {}
fitted_models = {}

for model_name, model in models.items():
    metrics, report, matrix = fit_and_evaluate(model_name, model)
    comparison_rows.append(metrics)
    classification_reports[model_name] = report
    confusion_matrices[model_name] = matrix
    fitted_models[model_name] = model  # In-memory only; never persisted.
    print(
        f"{model_name}: trained in {metrics['Training Seconds']:.2f}s; "
        f"validation prediction in "
        f"{metrics['Validation Prediction Seconds']:.3f}s"
    )

print('All baseline evaluations used validation only. Final test remains sealed.')

## 15. Compare models by validation Macro F1

**What:** build a comparison table sorted by the primary metric. **Why:** Macro F1 gives Poor, Standard, and Good equal importance even when class counts differ. **Expected output:** one row per model with quality and timing metrics. **Check:** higher validation Macro F1 is better, but also examine the train–validation gap and per-class results.

In [ ]:
comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values('Validation Macro F1', ascending=False)
    .reset_index(drop=True)
)
display(comparison.style.format({
    'Validation Macro F1': '{:.4f}',
    'Training Macro F1': '{:.4f}',
    'Train-Validation Gap': '{:.4f}',
    'Accuracy': '{:.4f}',
    'Balanced Accuracy': '{:.4f}',
    'Weighted F1': '{:.4f}',
    'Macro Precision': '{:.4f}',
    'Macro Recall': '{:.4f}',
    'Training Seconds': '{:.2f}',
    'Validation Prediction Seconds': '{:.3f}',
}))

## 16. Inspect per-class precision, recall, F1, and support

**What:** display a fixed-order report for Poor, Standard, and Good for every model. **Why:** an overall score can hide failure on one class. **Expected output:** four small tables. **Check:** look for a class with near-zero recall or F1, especially in DummyClassifier.

In [ ]:
for model_name in models:
    print(f'\n{model_name}')
    per_class = (
        pd.DataFrame(classification_reports[model_name])
        .T.loc[CLASS_ORDER, ['precision', 'recall', 'f1-score', 'support']]
    )
    display(per_class.style.format({
        'precision': '{:.4f}',
        'recall': '{:.4f}',
        'f1-score': '{:.4f}',
        'support': '{:.0f}',
    }))

## 17. Visualize confusion matrices

**What:** plot actual classes by predicted classes using the fixed Poor–Standard–Good order. **Why:** confusion matrices show which classes are confused with each other. **Expected output:** one labeled matrix per model. **Check:** strong models should have larger diagonal counts without ignoring a class.

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(13, 11))
for axis, model_name in zip(axes.ravel(), models):
    display_matrix = ConfusionMatrixDisplay(
        confusion_matrix=confusion_matrices[model_name],
        display_labels=CLASS_ORDER,
    )
    display_matrix.plot(ax=axis, cmap='Blues', colorbar=False, values_format='d')
    axis.set_title(model_name)
figure.suptitle('Validation confusion matrices — unseen customers', fontsize=15)
figure.tight_layout()
plt.show()

## 18. Beginner-friendly interpretation

**Underfitting** means a model is too simple to learn useful structure; both training and validation scores stay weak and close to the Dummy baseline. **Overfitting** means training performance is much better than validation performance, so the model may have memorized development patterns. The train–validation gap is a warning signal, not proof.

**What:** identify the strongest and weakest validation baselines and compare every learned model with DummyClassifier. **Expected output:** a short interpretation for each model. **Check:** do not choose a model from Macro F1 alone—also inspect class reports, confusion matrices, timing, and the gap. The 0.01 and 0.10 cutoffs below are teaching heuristics, not production acceptance rules.

In [ ]:
dummy_macro_f1 = comparison.loc[
    comparison['Model'].eq('DummyClassifier'), 'Validation Macro F1'
].iloc[0]
strongest = comparison.iloc[0]
weakest = comparison.iloc[-1]

print(
    f"Strongest validation baseline: {strongest['Model']} "
    f"(Macro F1={strongest['Validation Macro F1']:.4f})"
)
print(
    f"Weakest validation baseline: {weakest['Model']} "
    f"(Macro F1={weakest['Validation Macro F1']:.4f})"
)

for _, row in comparison.iterrows():
    model_name = row['Model']
    validation_score = row['Validation Macro F1']
    training_score = row['Training Macro F1']
    gap = row['Train-Validation Gap']
    if model_name == 'DummyClassifier':
        note = 'No-skill reference; expected to ignore minority classes.'
    elif gap > 0.10:
        note = 'Possible overfitting: training is much stronger than validation.'
    elif (
        validation_score <= dummy_macro_f1 + 0.01
        and training_score <= dummy_macro_f1 + 0.05
    ):
        note = 'Possible underfitting: both scores remain close to Dummy.'
    else:
        note = 'No large gap signal; still inspect every class before deciding.'
    beats_dummy = validation_score > dummy_macro_f1
    print(
        f"{model_name}: beats Dummy={beats_dummy}; gap={gap:.4f}. {note}"
    )

## 19. Final Phase 3A safety check

**What:** verify that no final-test transformed matrix or saved model artifact was created. **Why:** final test must remain untouched until model selection is finished in a later phase. **Expected output:** three explicit safety confirmations. **Check:** stop if any assertion fails; do not claim production readiness from these validation baselines.

In [ ]:
assert 'X_final_test_transformed' not in globals()
assert 'final_test_predictions' not in globals()
assert not any(REPOSITORY_DIR.glob('models/*phase3a*'))

print('Confirmed: final_test was not transformed, predicted, or evaluated.')
print('Confirmed: no model or transformed dataset was saved.')
print('Confirmed: these are validation baselines, not a production-ready model.')